# Dynamic Prompt
<img src="./assets/LC_DynamicPrompts.png" width="500">

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****954e
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass


@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT_TEMPLATE = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
{table_limits}
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

## Build a Dynamic Prompt
Utilize runtime context and middleware to generate a dynamic prompt.

In [6]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to these tables: Album, Artist, Genre, Playlist, PlaylistTrack, Track."
    else:
        table_limits = ""

    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

Include middleware in `create_agent`.

In [8]:
import os
from langchain_aws import ChatBedrock
from langchain.agents import create_agent

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    model_kwargs=dict(temperature=0),
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
)

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    #system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

In [9]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

[{'type': 'text', 'text': "<thinking> To identify the most costly purchase by Frank Harris, we need to execute a query that retrieves the maximum amount spent by him. This would require access to a database with a table that contains purchase records. Assuming the database is SQLite and there is a table called 'purchases' with columns 'buyer' and 'cost', I can construct a SQL query to retrieve this information. </thinking>\n"}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': "SELECT MAX(cost) AS Max_Purchase FROM purchases WHERE buyer='Frank Harris'"}, 'id': 'tooluse__OoLbyhCSBCdQy_Qmw62uA'}]
Tool Calls:
  execute_sql (tooluse__OoLbyhCSBCdQy_Qmw62uA)
 Call ID: tooluse__OoLbyhCSBCdQy_Qmw62uA
  Args:
    query: SELECT MAX(cost) AS Max_Purchase FROM purchases WHER

In [10]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

[{'type': 'text', 'text': '<thinking> I need to query the database to get the information about Frank Harris\' most costly purchase. I\'ll assume that there is a table named "purchases" and it has a column named "amount" for the cost of each purchase. The column "purchaser" will identify the person who made the purchase. I will execute a SQL query to retrieve the maximum amount from purchases made by Frank Harris. </thinking>\n'}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': "SELECT MAX(amount) as max_cost FROM purchases WHERE purchaser='Frank Harris'"}, 'id': 'tooluse_3nKWJqSWSWGy-RbdhZHqnA'}]
Tool Calls:
  execute_sql (tooluse_3nKWJqSWSWGy-RbdhZHqnA)
 Call ID: tooluse_3nKWJqSWSWGy-RbdhZHqnA
  Args:
    query: SELECT MAX(amount) as max_cost FROM purchases W